# SBD Modular Architecture — GSC Experiments

End-to-end notebook for training and evaluating the Sparse Binary Distributed (SBD)
modular architecture on the Google Speech Commands v2 dataset.

**Assumptions**
- `dataset.py` and `model.py` reside in the same directory as this notebook.
- `GSC_ROOT` below points to the extracted GSC v2 corpus root (the folder
  containing `validation_list.txt`, `testing_list.txt`, and the per-class
  subdirectories).

In [ ]:
# ── Cell 1: Imports ─────────────────────────────────────────────────────────

# Standard library
import os
import random
from pathlib import Path

# Numerical / plotting
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# Project modules (assumed to live in the same directory)
from dataset import (
    GSCDataset,
    SpecAugmentTransform,
    GSC_CLASSES,
    N_MELS,
    N_FRAMES,
)
from model import (
    SBDConfig,
    SBDStack,
    LinearClassifier,
    build_model,
    CAInterface,
    SBDEncoder,
    SBDModule,
    make_watts_strogatz_graph,
    make_fixed_fanin_graph,
    make_fixed_fanin_bipartite,
    ca_and_or_step,
)

# ── Reproducibility seed ────────────────────────────────────────────────────
GLOBAL_SEED = 42

random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
torch.cuda.manual_seed_all(GLOBAL_SEED)
# Deterministic algorithms where available; may be slower on some ops.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# ── Device ──────────────────────────────────────────────────────────────────
DEVICE = torch.device(
    "cuda"  if torch.cuda.is_available()  else
    "mps"   if torch.backends.mps.is_available() else
    "cpu"
)
print(f"Using device : {DEVICE}")
print(f"PyTorch      : {torch.__version__}")
print(f"Classes      : {len(GSC_CLASSES)}")

In [ ]:
# ── Cell 2: Dataset and DataLoaders ─────────────────────────────────────────
#
# Two speed knobs are available:
#
#   MAX_SAMPLES  — set to an integer (e.g. 500) to load only the first N
#                  samples from each split.  Useful for a quick smoke-test.
#                  Set to None for the full corpus.
#
#   CACHE_DIR    — directory where pre-computed .pt cache files are written.
#                  On the first run the cache is built once and saved (~10 min
#                  for the full corpus).  Every subsequent run loads it in
#                  seconds.  Set to None to disable disk caching (RAM only).
#
# Typical dev workflow:
#   1. First pass  : MAX_SAMPLES=500, CACHE_DIR=None  → instant, smoke-test
#   2. Full run    : MAX_SAMPLES=None, CACHE_DIR="./cache"  → saves to disk
#   3. Reruns      : MAX_SAMPLES=None, CACHE_DIR="./cache"  → loads in ~5 s

MAX_SAMPLES: int | None = 500 # None          # ← set to e.g. 500 for quick testing
CACHE_DIR:   str | None = "./cache"     # ← set to None to skip disk caching

# ── Path to the extracted GSC v2 corpus ─────────────────────────────────────
GSC_ROOT = Path("/home/iori/research/infomax-sbdr/resources/datasets/gsc")   # ← update before running

# ── Dataset construction ─────────────────────────────────────────────────────
train_dataset = GSCDataset(
    root        = GSC_ROOT,
    split       = "train",
    precompute  = True,
    augment     = SpecAugmentTransform(
        max_time_width=8,
    ),
    max_samples = MAX_SAMPLES,
    cache_dir   = CACHE_DIR,
)

val_dataset = GSCDataset(
    root        = GSC_ROOT,
    split       = "val",
    precompute  = True,
    augment     = None,
    max_samples = MAX_SAMPLES,
    cache_dir   = CACHE_DIR,
)

test_dataset = GSCDataset(
    root        = GSC_ROOT,
    split       = "test",
    precompute  = True,
    augment     = None,
    max_samples = MAX_SAMPLES,
    cache_dir   = CACHE_DIR,
)

# ── Reproducible DataLoader generator ────────────────────────────────────────
dl_generator = torch.Generator()
dl_generator.manual_seed(GLOBAL_SEED)

# ── DataLoaders ──────────────────────────────────────────────────────────────
BATCH_SIZE  = 256
NUM_WORKERS = min(4, os.cpu_count() or 1)

train_loader = DataLoader(
    train_dataset,
    batch_size  = BATCH_SIZE,
    shuffle     = True,
    num_workers = NUM_WORKERS,
    pin_memory  = DEVICE.type == "cuda",
    drop_last   = True,
    generator   = dl_generator,
)

val_loader = DataLoader(
    val_dataset,
    batch_size  = BATCH_SIZE,
    shuffle     = False,
    num_workers = NUM_WORKERS,
    pin_memory  = DEVICE.type == "cuda",
)

test_loader = DataLoader(
    test_dataset,
    batch_size  = BATCH_SIZE,
    shuffle     = False,
    num_workers = NUM_WORKERS,
    pin_memory  = DEVICE.type == "cuda",
)

# ── Summary ──────────────────────────────────────────────────────────────────
print(f"Train samples : {len(train_dataset):>7,}   batches: {len(train_loader):,}")
print(f"Val   samples : {len(val_dataset):>7,}   batches: {len(val_loader):,}")
print(f"Test  samples : {len(test_dataset):>7,}   batches: {len(test_loader):,}")
print(f"Batch size    : {BATCH_SIZE}")
print(f"Spectrogram   : ({N_MELS} mels × {N_FRAMES} frames)")
print(f"Cache dir     : {CACHE_DIR}")

In [ ]:
# ── Cell 3: Visualise a data sample ─────────────────────────────────────────
#
# Displays one raw (pre-augment) and one augmented spectrogram side-by-side,
# together with the per-mel-bin amplitude distribution across the training set
# (sampled from the first batch) to verify CMVN normalisation.

# ── Pick a reproducible sample ───────────────────────────────────────────────
SAMPLE_IDX = 0   # change to inspect a different utterance

# Fetch from the cache (no augment) by temporarily bypassing SpecAugment.
_orig_augment          = train_dataset.augment
train_dataset.augment  = None
spec_clean, label      = train_dataset[SAMPLE_IDX]   # (n_mels, n_frames)
train_dataset.augment  = _orig_augment

# Fetch with augmentation.
spec_augmented, _      = train_dataset[SAMPLE_IDX]   # (n_mels, n_frames)

class_name = GSC_CLASSES[label]

# ── Figure 1: clean vs augmented spectrogram ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, spec, title in zip(
    axes,
    [spec_clean, spec_augmented],
    ["Clean (CMVN-normalised)", "SpecAugmented"],
):
    # spec shape: (n_mels, n_frames) — imshow expects (rows, cols)
    # origin='lower' puts mel bin 0 at the bottom (conventional).
    im = ax.imshow(
        spec.numpy(),
        aspect    = "auto",
        origin    = "lower",
        interpolation = "nearest",
        cmap      = "magma",
    )
    ax.set_title(f"{title}\nlabel: '{class_name}' (idx {label})",
                 fontsize=11)
    ax.set_xlabel("Frame index (10 ms / frame)", fontsize=9)
    ax.set_ylabel("Mel bin", fontsize=9)
    # Tick every 10 frames (= 100 ms).
    ax.xaxis.set_major_locator(ticker.MultipleLocator(10))
    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label="Normalised value")

fig.suptitle(
    f"Sample {SAMPLE_IDX} — Log-Mel spectrogram ({N_MELS} bins × {N_FRAMES} frames)",
    fontsize=13, y=1.02,
)
plt.tight_layout()
plt.show()

# ── Figure 2: CMVN sanity check over first batch ─────────────────────────────
# After CMVN each mel bin should have mean ≈ 0 and std ≈ 1 *per utterance*.
# Across utterances the across-batch distribution will have non-zero variance
# in its means and stds — that is expected and correct.

_loader_iter            = iter(val_loader)
batch_specs, batch_lbls = next(_loader_iter)   # (B, n_mels, n_frames)

# Per-utterance, per-mel-bin mean and std.
per_utt_mean = batch_specs.mean(dim=-1)   # (B, n_mels)
per_utt_std  = batch_specs.std( dim=-1)   # (B, n_mels)

fig2, axes2 = plt.subplots(1, 2, figsize=(12, 3))

axes2[0].imshow(
    per_utt_mean.numpy().T,
    aspect="auto", origin="lower", cmap="RdBu_r", interpolation="nearest",
    vmin=-1.0, vmax=1.0,
)
axes2[0].set_title("Per-utterance CMVN mean (should be ≈ 0 per row)",
                   fontsize=10)
axes2[0].set_xlabel("Utterance in batch", fontsize=9)
axes2[0].set_ylabel("Mel bin", fontsize=9)

axes2[1].imshow(
    per_utt_std.numpy().T,
    aspect="auto", origin="lower", cmap="viridis", interpolation="nearest",
    vmin=0.0, vmax=2.0,
)
axes2[1].set_title("Per-utterance CMVN std (should be ≈ 1 per row)",
                   fontsize=10)
axes2[1].set_xlabel("Utterance in batch", fontsize=9)
axes2[1].set_ylabel("Mel bin", fontsize=9)

fig2.suptitle(
    f"CMVN sanity check — first validation batch (B={batch_specs.shape[0]})",
    fontsize=12, y=1.02,
)
plt.tight_layout()
plt.show()

# ── Numerical summary ────────────────────────────────────────────────────────
print("Batch spectrogram shape :", tuple(batch_specs.shape))
print(f"Per-utterance mean  — grand mean: {per_utt_mean.mean():.4f}, "
      f"std of means: {per_utt_mean.std():.4f}   (expect ≈ 0, small)")
print(f"Per-utterance std   — grand mean: {per_utt_std.mean():.4f}, "
      f"std of stds : {per_utt_std.std():.4f}   (expect ≈ 1, small)")

# ── Per-class count in the first batch ───────────────────────────────────────
unique_lbls, counts = batch_lbls.unique(return_counts=True)
print("\nClass distribution in first val batch:")
for lbl, cnt in sorted(zip(unique_lbls.tolist(), counts.tolist()),
                       key=lambda x: -x[1]):
    print(f"  {GSC_CLASSES[lbl]:>10s}  ({lbl:2d})  : {cnt}")

In [ ]:
# ── Cell 4: Build architecture and smoke-test forward pass ───────────────────

# ── Config ───────────────────────────────────────────────────────────────────
cfg = SBDConfig(
    n_input    = N_MELS,   # 40 mel bins
    n_channels = 128,      # binary units per encoder
    n_hidden   = 128,      # CA hidden units
    n_layers   = 2,        # start small: one continuous-input + one binary
    k_i        = 3,        # fan-in from input population to CA hidden units
    k_h        = 3,        # fan-in from hidden population to CA hidden units
    topology   = "small_world",
    sw_beta    = 0.1,
    layer_norm = True,
    seed       = GLOBAL_SEED,
)

stack, classifier = build_model(cfg)
stack      = stack.to(DEVICE)
classifier = classifier.to(DEVICE)

# ── Parameter count ──────────────────────────────────────────────────────────
def count_parameters(module: nn.Module, trainable_only: bool = True) -> int:
    return sum(
        p.numel() for p in module.parameters()
        if (p.requires_grad or not trainable_only)
    )

print("── Architecture ────────────────────────────────────────────────────")
print(stack)
print()
print("── Parameter counts ────────────────────────────────────────────────")
for i, layer in enumerate(stack.layers):
    n = count_parameters(layer, trainable_only=False)
    label = "(continuous input, no CA)" if i == 0 else "(binary input + CA)"
    print(f"  Layer {i} {label}: {n:,} params")
print(f"  Stack total (learned)  : {count_parameters(stack):,}")
print(f"  Classifier             : {count_parameters(classifier):,}")

# ── Grab one real batch from the val loader ───────────────────────────────────
spec_batch, label_batch = next(iter(val_loader))
spec_batch  = spec_batch.to(DEVICE)    # (B, N_MELS, T)
label_batch = label_batch.to(DEVICE)

print()
print("── Input ───────────────────────────────────────────────────────────")
print(f"  spec_batch  : {tuple(spec_batch.shape)}  dtype={spec_batch.dtype}")
print(f"  label_batch : {tuple(label_batch.shape)} dtype={label_batch.dtype}")

# ── Full forward pass ─────────────────────────────────────────────────────────
stack.eval()
with torch.no_grad():
    outputs = stack(spec_batch)          # list of (a, z) per layer

print()
print("── Per-layer outputs ────────────────────────────────────────────────")
for i, (a, z) in enumerate(outputs):
    # Empirical sparsity: fraction of True entries in z
    sparsity = z.float().mean().item()
    print(
        f"  Layer {i} │ "
        f"a: {tuple(a.shape)} {a.dtype} │ "
        f"z: {tuple(z.shape)} {z.dtype} │ "
        f"sparsity={sparsity:.3f}  "
        f"(target ≈ 0.150)"
    )

# ── Classifier forward pass ───────────────────────────────────────────────────
# Use the final layer's binary output, global-average-pooled over time.
with torch.no_grad():
    final_z  = outputs[-1][1]            # (B, T, C) bool
    logits   = classifier([final_z])     # (B, n_classes)
    preds    = logits.argmax(dim=-1)     # (B,)

print()
print("── Classifier ───────────────────────────────────────────────────────")
print(f"  logits : {tuple(logits.shape)}  dtype={logits.dtype}")
print(f"  preds  : {tuple(preds.shape)}")

# ── CA hidden state sparsity across layers ────────────────────────────────────
# Verify that the CA interface produces the expected ~0.149 output sparsity.
print()
print("── CA hidden state sparsity check ──────────────────────────────────")
for i, layer in enumerate(stack.layers):
    if layer.ca is None:
        print(f"  Layer {i} │ no CA (first module)")
        continue
    # Run the CA in isolation on the binary output of the previous layer.
    prev_z = outputs[i - 1][1]          # (B, T, C) bool
    with torch.no_grad():
        zeta = layer.ca(prev_z)          # (B, T, C + C_h) bool
    h = zeta[..., cfg.n_channels:]      # (B, T, C_h) — hidden part only
    print(
        f"  Layer {i} │ "
        f"CA output (hidden) sparsity={h.float().mean().item():.3f}  "
        f"(analytical prediction ≈ {(1-(1-0.15)**cfg.k_i)**2:.3f})"
    )

print()
print("── All checks passed ────────────────────────────────────────────────")